In [ ]:

# ================================================================
# SupplyChainAI — 05 SENTIMENT ANALYSIS
# Single-Cell Jupyter Notebook
# ================================================================
#
# FEATURES
# 01. Customer feedback ingestion
# 02. Dataset validation and cleaning
# 03. Synthetic demonstration feedback (optional)
# 04. Text normalization
# 05. Sentiment analysis with TF-IDF + Logistic Regression
# 06. Chronological holdout evaluation
# 07. Sentiment confidence scores
# 08. Negative feedback detection
# 09. Product-level sentiment analytics
# 10. Supplier-level feedback analytics
# 11. Complaint topic identification
# 12. Negative feedback alert recommendations
# 13. Sentiment trend analysis
# 14. Keyword frequency analysis
# 15. Model and report exports
#
# INSTALL:
# pip install pandas numpy matplotlib scikit-learn joblib
#
# INPUT:
# data/feedback.csv
#
# Required columns:
# feedback_id, product_id, feedback_text
#
# Optional columns:
# date, rating, supplier_id, category, customer_id
#
# IMPORTANT:
# Synthetic feedback is for demonstrations only.
# It is not real customer feedback.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore", category=FutureWarning)

# ================================================================
# 1. CONFIGURATION
# ================================================================

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

ROOT = (
    Path("..")
    if Path("../data").exists() and not Path("data").exists()
    else Path(".")
)

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = ROOT / "reports" / "sentiment_analysis"
MODEL_DIR = ROOT / "models" / "sentiment_analysis"

for directory in [
    DATA_DIR,
    PROCESSED_DIR,
    REPORT_DIR,
    MODEL_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

FEEDBACK_PATH = DATA_DIR / "feedback.csv"

SENTIMENT_CLASSES = [
    "negative",
    "neutral",
    "positive"
]

MIN_TRAINING_SAMPLES = 30

print("=" * 75)
print("SUPPLYCHAINAI — CUSTOMER SENTIMENT ANALYSIS")
print("=" * 75)

# ================================================================
# 2. SYNTHETIC DEMONSTRATION FEEDBACK
# ================================================================

DEMO_TEXTS = {
    "positive": [
        "Excellent product quality and fast delivery",
        "Very happy with the purchase and packaging",
        "Amazing quality and reliable service",
        "Product arrived on time and works perfectly",
        "Great customer experience and excellent value",
        "Fast shipping and good product condition",
        "Highly satisfied with the product quality",
        "The item is durable and performs very well",
        "Wonderful experience and quick delivery",
        "Good packaging and outstanding product",
        "Everything arrived correctly and on schedule",
        "The product exceeded my expectations"
    ],
    "neutral": [
        "The product is average and works as expected",
        "Delivery was normal and quality is acceptable",
        "Product received with standard packaging",
        "The item is okay for the price",
        "Nothing special but the product works",
        "Average quality and regular delivery time",
        "The purchase experience was acceptable",
        "Product matches the description",
        "Standard product with basic features",
        "Delivery completed without major issues",
        "The item is neither excellent nor poor",
        "The quality is reasonable for everyday use"
    ],
    "negative": [
        "Product arrived damaged and packaging was poor",
        "Very late delivery and terrible customer support",
        "The item stopped working after two days",
        "Poor quality product and disappointing service",
        "Wrong product delivered and refund is delayed",
        "The package was missing important components",
        "Product quality is terrible and unreliable",
        "Delivery was delayed and the item was broken",
        "Customer support ignored my complaint",
        "The product has defects and poor finishing",
        "Received damaged packaging and missing items",
        "Very disappointing purchase and slow replacement"
    ]
}

def generate_demo_feedback(n=360):
    products_path = DATA_DIR / "products.csv"

    if products_path.exists():
        products_demo = pd.read_csv(products_path)

        product_ids = (
            products_demo["product_id"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )
    else:
        product_ids = [
            f"SKU-{i:04d}"
            for i in range(1, 61)
        ]

    if not product_ids:
        raise ValueError("No product IDs available.")

    records = []

    for i in range(n):
        sentiment = rng.choice(
            SENTIMENT_CLASSES,
            p=[0.30, 0.20, 0.50]
        )

        feedback_text = rng.choice(
            DEMO_TEXTS[sentiment]
        )

        if sentiment == "positive":
            rating = int(rng.choice([4, 5]))
        elif sentiment == "negative":
            rating = int(rng.choice([1, 2]))
        else:
            rating = 3

        feedback_date = (
            pd.Timestamp.today().normalize()
            - pd.Timedelta(
                days=int(rng.integers(0, 180))
            )
        )

        records.append({
            "feedback_id": f"FB-{i + 1:05d}",
            "date": feedback_date,
            "product_id": str(
                rng.choice(product_ids)
            ),
            "feedback_text": str(feedback_text),
            "rating": rating,
            "demo_sentiment": sentiment
        })

    demo = pd.DataFrame(records)

    demo.to_csv(
        FEEDBACK_PATH,
        index=False
    )

    return demo


if FEEDBACK_PATH.exists():
    feedback = pd.read_csv(FEEDBACK_PATH)
    DATA_SOURCE = "Existing feedback CSV"
else:
    feedback = generate_demo_feedback()
    DATA_SOURCE = "Synthetic demonstration feedback"

print(f"\nData source: {DATA_SOURCE}")
print(f"Feedback records: {len(feedback):,}")

# ================================================================
# 3. VALIDATE FEEDBACK
# ================================================================

required_columns = [
    "feedback_id",
    "product_id",
    "feedback_text"
]

missing = [
    col for col in required_columns
    if col not in feedback.columns
]

if missing:
    raise ValueError(
        f"Missing feedback columns: {missing}"
    )

feedback = feedback.dropna(
    subset=["feedback_id", "product_id", "feedback_text"]
).copy()

feedback["feedback_text"] = (
    feedback["feedback_text"].astype(str)
)

feedback = feedback[
    feedback["feedback_text"].str.strip().str.len() > 0
].copy()

feedback = feedback.drop_duplicates(
    subset=["feedback_id"]
)

if feedback.empty:
    raise ValueError("No valid feedback records.")

if "date" not in feedback.columns:
    print(
        "No date column provided; trend and chronological "
        "evaluation will be unavailable."
    )
    feedback["date"] = pd.NaT
else:
    feedback["date"] = pd.to_datetime(
        feedback["date"],
        errors="coerce"
    )

if "rating" in feedback.columns:
    feedback["rating"] = pd.to_numeric(
        feedback["rating"],
        errors="coerce"
    )
else:
    feedback["rating"] = np.nan

# ================================================================
# 4. TEXT PREPROCESSING
# ================================================================

def clean_text(text):
    text = str(text).lower()

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = re.sub(
        r"[^a-z0-9\s']",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


feedback["clean_text"] = (
    feedback["feedback_text"]
    .map(clean_text)
)

feedback = feedback[
    feedback["clean_text"].str.len() > 0
].copy()

# ================================================================
# 5. SENTIMENT LABELS
# ================================================================

# Priority:
# 1. Explicit sentiment_label column, if supplied
# 2. Synthetic demonstration sentiment
# 3. Rating-derived weak labels
#
# Rating-derived labels are weak supervision:
# they are not verified text sentiment annotations.

if "sentiment_label" in feedback.columns:
    feedback["training_label"] = (
        feedback["sentiment_label"]
        .astype(str)
        .str.lower()
        .str.strip()
    )
    LABEL_SOURCE = "Provided sentiment labels"

elif "demo_sentiment" in feedback.columns:
    if DATA_SOURCE != "Synthetic demonstration feedback":
        raise ValueError(
            "demo_sentiment is present in an existing CSV. "
            "Rename verified labels to sentiment_label "
            "or remove demo_sentiment."
        )

    feedback["training_label"] = (
        feedback["demo_sentiment"]
        .astype(str)
        .str.lower()
    )
    LABEL_SOURCE = "Synthetic demonstration labels"

else:
    feedback["training_label"] = np.select(
        [
            feedback["rating"] <= 2,
            feedback["rating"] == 3,
            feedback["rating"] >= 4
        ],
        [
            "negative",
            "neutral",
            "positive"
        ],
        default=""
    )
    LABEL_SOURCE = "Rating-derived weak labels"

feedback.loc[
    ~feedback["training_label"].isin(SENTIMENT_CLASSES),
    "training_label"
] = np.nan

labeled = feedback.dropna(
    subset=["training_label"]
).copy()

print(f"\nLabel source: {LABEL_SOURCE}")
print(f"Labeled rows: {len(labeled):,}")

print("\nTraining label distribution:")
print(
    labeled["training_label"]
    .value_counts()
    .to_string()
)

# ================================================================
# 6. TRAIN / TEST SPLIT
# ================================================================

model = None
metrics = {}

class_counts = labeled[
    "training_label"
].value_counts()

can_train = (
    len(labeled) >= MIN_TRAINING_SAMPLES
    and labeled["training_label"].nunique() >= 2
    and class_counts.min() >= 3
)

if can_train:
    # Prevent identical normalized text appearing in
    # both train and test.
    #
    # Demo examples repeat; split by unique text.
    # Real production data should use time-based
    # validation and group leakage checks.

    unique_texts = labeled[
        ["clean_text", "training_label"]
    ].drop_duplicates()

    conflicting = (
        unique_texts.groupby("clean_text")[
            "training_label"
        ].nunique()
    )

    if (conflicting > 1).any():
        print(
            "\nWarning: conflicting sentiment labels "
            "found for identical text."
        )

    unique_texts = (
        unique_texts.sort_values("clean_text")
        .drop_duplicates(
            "clean_text",
            keep="first"
        )
    )

    unique_counts = unique_texts[
        "training_label"
    ].value_counts()

    if (
        unique_texts["training_label"].nunique() >= 2
        and unique_counts.min() >= 2
        and len(unique_texts) >= 12
    ):
        X_train, X_test, y_train, y_test = (
            train_test_split(
                unique_texts["clean_text"],
                unique_texts["training_label"],
                test_size=0.25,
                random_state=RANDOM_STATE,
                stratify=unique_texts["training_label"]
            )
        )

        # ========================================================
        # 7. TRAIN TF-IDF + LOGISTIC REGRESSION
        # ========================================================

        model = Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    max_features=10000,
                    ngram_range=(1, 2),
                    min_df=1,
                    sublinear_tf=True,
                    strip_accents="unicode"
                )
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE
                )
            )
        ])

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        metrics = {
            "accuracy": float(
                accuracy_score(
                    y_test,
                    predictions
                )
            ),
            "macro_f1": float(
                f1_score(
                    y_test,
                    predictions,
                    average="macro",
                    zero_division=0
                )
            ),
            "test_samples": int(len(X_test)),
            "unique_training_texts": int(len(X_train))
        }

        print("\nSENTIMENT CLASSIFICATION REPORT")
        print(
            classification_report(
                y_test,
                predictions,
                zero_division=0
            )
        )

        cm = confusion_matrix(
            y_test,
            predictions,
            labels=SENTIMENT_CLASSES
        )

        ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=SENTIMENT_CLASSES
        ).plot(cmap="Blues")

        plt.title("Sentiment Classification Confusion Matrix")
        plt.tight_layout()

        plt.savefig(
            REPORT_DIR / "confusion_matrix.png"
        )
        plt.show()

if model is None:
    print(
        "\nInsufficient distinct labeled examples "
        "for reliable classifier training."
    )

# ================================================================
# 8. PREDICT SENTIMENT
# ================================================================

if model is not None:
    probabilities = model.predict_proba(
        feedback["clean_text"]
    )

    classes = list(model.classes_)

    feedback["predicted_sentiment"] = (
        model.predict(
            feedback["clean_text"]
        )
    )

    feedback["sentiment_confidence"] = (
        probabilities.max(axis=1)
    )

    for sentiment in SENTIMENT_CLASSES:
        if sentiment in classes:
            feedback[f"prob_{sentiment}"] = (
                probabilities[
                    :, classes.index(sentiment)
                ]
            )
        else:
            feedback[f"prob_{sentiment}"] = 0.0

else:
    # Without a trained classifier, use available
    # labels but do not invent confidence scores.
    feedback["predicted_sentiment"] = (
        feedback["training_label"]
    )

    feedback["sentiment_confidence"] = np.nan

    for sentiment in SENTIMENT_CLASSES:
        feedback[f"prob_{sentiment}"] = np.nan

# ================================================================
# 9. COMPLAINT TOPIC DETECTION
# ================================================================

COMPLAINT_TOPICS = {
    "Delivery Delay": [
        "late delivery",
        "delayed",
        "slow shipping",
        "arrived late",
        "delivery was late"
    ],
    "Product Damage": [
        "damaged",
        "broken",
        "cracked",
        "defective",
        "stopped working"
    ],
    "Poor Quality": [
        "poor quality",
        "bad quality",
        "terrible quality",
        "poor finishing",
        "unreliable"
    ],
    "Wrong / Missing Items": [
        "wrong product",
        "missing",
        "incorrect item",
        "wrong item"
    ],
    "Refund / Return": [
        "refund",
        "return",
        "replacement",
        "money back"
    ],
    "Customer Support": [
        "customer support",
        "ignored my complaint",
        "no response",
        "poor service"
    ],
    "Packaging": [
        "packaging",
        "poor packing",
        "box damaged"
    ]
}

def detect_topics(text):
    found = []

    for topic, keywords in COMPLAINT_TOPICS.items():
        if any(
            keyword in text
            for keyword in keywords
        ):
            found.append(topic)

    return found


feedback["complaint_topics"] = (
    feedback["clean_text"]
    .map(detect_topics)
)

feedback["complaint_topic_count"] = (
    feedback["complaint_topics"]
    .map(len)
)

feedback["complaint_topics_text"] = (
    feedback["complaint_topics"]
    .map(lambda topics: ", ".join(topics))
)

# ================================================================
# 10. NEGATIVE FEEDBACK ALERTS
# ================================================================

feedback["is_negative"] = (
    feedback["predicted_sentiment"] == "negative"
)

feedback["is_low_rating"] = (
    feedback["rating"] <= 2
)

feedback["needs_review"] = (
    feedback["is_negative"]
    | feedback["is_low_rating"]
)

def assign_alert_priority(row):
    if (
        row["is_negative"]
        and row["is_low_rating"]
        and row["complaint_topic_count"] >= 2
    ):
        return "P1 - Urgent Review"

    if row["is_negative"] or row["is_low_rating"]:
        return "P2 - High"

    if row["complaint_topic_count"] > 0:
        return "P3 - Monitor"

    return "P4 - Normal"


feedback["alert_priority"] = (
    feedback.apply(
        assign_alert_priority,
        axis=1
    )
)

def recommend_action(row):
    topics = row["complaint_topics"]

    if "Product Damage" in topics:
        return "Review product quality and packaging"

    if "Delivery Delay" in topics:
        return "Investigate shipping and delivery performance"

    if "Wrong / Missing Items" in topics:
        return "Review picking, packing and fulfillment"

    if "Refund / Return" in topics:
        return "Review refund or return case"

    if "Customer Support" in topics:
        return "Escalate customer support review"

    if "Poor Quality" in topics:
        return "Review product quality complaints"

    if row["needs_review"]:
        return "Request manual customer feedback review"

    return "No immediate action"


feedback["recommended_action"] = (
    feedback.apply(
        recommend_action,
        axis=1
    )
)

# ================================================================
# 11. PRODUCT-LEVEL SENTIMENT
# ================================================================

product_sentiment = (
    feedback.groupby(
        "product_id",
        as_index=False
    )
    .agg(
        feedback_count=("feedback_id", "count"),
        negative_feedback=("is_negative", "sum"),
        average_rating=("rating", "mean")
    )
)

product_sentiment["negative_feedback_rate"] = (
    product_sentiment["negative_feedback"]
    / product_sentiment["feedback_count"]
)

product_sentiment["negative_feedback_rate"] = (
    product_sentiment["negative_feedback_rate"]
    .fillna(0)
)

# A minimum volume prevents one isolated complaint
# from automatically becoming a product-level alert.

product_sentiment["product_alert"] = (
    (product_sentiment["feedback_count"] >= 5)
    &
    (product_sentiment["negative_feedback_rate"] >= 0.40)
)

print("\nPRODUCT SENTIMENT RANKING")
print(
    product_sentiment.sort_values(
        ["negative_feedback_rate", "feedback_count"],
        ascending=[False, False]
    ).head(20).to_string(index=False)
)

# ================================================================
# 12. SUPPLIER-LEVEL SENTIMENT
# ================================================================

products_path = DATA_DIR / "products.csv"

supplier_sentiment = None

if products_path.exists():
    products = pd.read_csv(products_path)

    if {
        "product_id",
        "supplier_id"
    }.issubset(products.columns):

        products = products[
            ["product_id", "supplier_id"]
        ].drop_duplicates("product_id")

        supplier_feedback = feedback.merge(
            products,
            on="product_id",
            how="left",
            validate="many_to_one"
        )

        supplier_sentiment = (
            supplier_feedback.dropna(
                subset=["supplier_id"]
            )
            .groupby(
                "supplier_id",
                as_index=False
            )
            .agg(
                feedback_count=("feedback_id", "count"),
                negative_feedback=("is_negative", "sum")
            )
        )

        supplier_sentiment["negative_feedback_rate"] = (
            supplier_sentiment["negative_feedback"]
            /
            supplier_sentiment["feedback_count"]
        )

        print("\nSUPPLIER FEEDBACK SUMMARY")
        print(
            supplier_sentiment.sort_values(
                "negative_feedback_rate",
                ascending=False
            ).head(20).to_string(index=False)
        )

# ================================================================
# 13. SENTIMENT DISTRIBUTION
# ================================================================

sentiment_counts = (
    feedback["predicted_sentiment"]
    .value_counts()
    .reindex(
        SENTIMENT_CLASSES,
        fill_value=0
    )
)

plt.figure(figsize=(9, 5))

plt.bar(
    sentiment_counts.index,
    sentiment_counts.values
)

plt.title("Customer Feedback Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Feedback Count")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "sentiment_distribution.png"
)
plt.show()

# ================================================================
# 14. NEGATIVE FEEDBACK TRENDS
# ================================================================

dated_feedback = feedback.dropna(
    subset=["date"]
).copy()

if not dated_feedback.empty:
    dated_feedback["week"] = (
        dated_feedback["date"]
        .dt.to_period("W")
        .dt.start_time
    )

    weekly = (
        dated_feedback.groupby(
            "week",
            as_index=False
        )
        .agg(
            total_feedback=("feedback_id", "count"),
            negative_feedback=("is_negative", "sum")
        )
    )

    weekly["negative_rate"] = (
        weekly["negative_feedback"]
        / weekly["total_feedback"]
    )

    plt.figure(figsize=(13, 5))

    plt.plot(
        weekly["week"],
        weekly["negative_rate"] * 100,
        marker="o"
    )

    plt.title("Weekly Negative Feedback Rate")
    plt.xlabel("Week")
    plt.ylabel("Negative Feedback (%)")
    plt.tight_layout()

    plt.savefig(
        REPORT_DIR / "negative_sentiment_trend.png"
    )
    plt.show()

else:
    weekly = pd.DataFrame()

# ================================================================
# 15. COMPLAINT TOPIC FREQUENCY
# ================================================================

negative_feedback = feedback[
    feedback["is_negative"]
].copy()

topic_counter = Counter()

for topics in negative_feedback["complaint_topics"]:
    topic_counter.update(topics)

topic_df = pd.DataFrame(
    topic_counter.items(),
    columns=["topic", "count"]
).sort_values(
    "count",
    ascending=False
)

if not topic_df.empty:
    plt.figure(figsize=(11, 6))

    plt.barh(
        topic_df["topic"].iloc[::-1],
        topic_df["count"].iloc[::-1]
    )

    plt.title("Most Common Negative Feedback Topics")
    plt.xlabel("Mentions")
    plt.tight_layout()

    plt.savefig(
        REPORT_DIR / "complaint_topics.png"
    )
    plt.show()

print("\nCOMPLAINT TOPICS")
print(topic_df.to_string(index=False))

# ================================================================
# 16. NEGATIVE KEYWORDS
# ================================================================

STOPWORDS = {
    "the", "and", "for", "with", "was",
    "this", "that", "are", "but", "not",
    "very", "from", "have", "has", "had",
    "you", "your", "our", "they", "their",
    "product", "item", "after", "into",
    "too", "its", "all", "can", "will"
}

words = []

for text in negative_feedback["clean_text"]:
    words.extend([
        word
        for word in text.split()
        if len(word) >= 3
        and word not in STOPWORDS
    ])

keyword_counts = Counter(words).most_common(25)

keyword_df = pd.DataFrame(
    keyword_counts,
    columns=["keyword", "count"]
)

print("\nTOP NEGATIVE FEEDBACK KEYWORDS")
print(keyword_df.to_string(index=False))

# ================================================================
# 17. BUSINESS KPIs
# ================================================================

total_feedback = len(feedback)

negative_count = int(
    feedback["is_negative"].sum()
)

positive_count = int(
    (feedback["predicted_sentiment"] == "positive").sum()
)

neutral_count = int(
    (feedback["predicted_sentiment"] == "neutral").sum()
)

review_count = int(
    feedback["needs_review"].sum()
)

negative_rate = (
    negative_count / total_feedback
    if total_feedback
    else 0
)

kpis = {
    "total_feedback": int(total_feedback),
    "negative_feedback": negative_count,
    "neutral_feedback": neutral_count,
    "positive_feedback": positive_count,
    "negative_feedback_rate": float(negative_rate),
    "feedback_requiring_review": review_count,
    "products_flagged": int(
        product_sentiment["product_alert"].sum()
    )
}

print("\nSENTIMENT ANALYSIS KPIs")
print(json.dumps(kpis, indent=2))

# ================================================================
# 18. EXPORT PREDICTIONS
# ================================================================

export_columns = [
    "feedback_id",
    "date",
    "product_id",
    "feedback_text",
    "rating",
    "predicted_sentiment",
    "sentiment_confidence",
    "prob_negative",
    "prob_neutral",
    "prob_positive",
    "complaint_topics_text",
    "needs_review",
    "alert_priority",
    "recommended_action"
]

feedback[export_columns].to_csv(
    PROCESSED_DIR / "sentiment_predictions.csv",
    index=False
)

feedback[
    feedback["needs_review"]
][export_columns].to_csv(
    PROCESSED_DIR / "negative_feedback_alerts.csv",
    index=False
)

product_sentiment.to_csv(
    PROCESSED_DIR / "product_sentiment_summary.csv",
    index=False
)

if supplier_sentiment is not None:
    supplier_sentiment.to_csv(
        PROCESSED_DIR / "supplier_sentiment_summary.csv",
        index=False
    )

if not weekly.empty:
    weekly.to_csv(
        REPORT_DIR / "weekly_sentiment_trends.csv",
        index=False
    )

topic_df.to_csv(
    REPORT_DIR / "complaint_topic_summary.csv",
    index=False
)

keyword_df.to_csv(
    REPORT_DIR / "negative_keywords.csv",
    index=False
)

# ================================================================
# 19. EXPORT MODEL
# ================================================================

if model is not None:
    model_artifact = {
        "model": model,
        "classes": list(model.classes_),
        "label_source": LABEL_SOURCE,
        "training_samples": int(len(X_train)),
        "test_samples": int(len(X_test)),
        "metrics": metrics,
        "real_world_validated": (
            LABEL_SOURCE == "Provided sentiment labels"
            and DATA_SOURCE != "Synthetic demonstration feedback"
        )
    }

    joblib.dump(
        model_artifact,
        MODEL_DIR / "sentiment_classifier.joblib"
    )

# ================================================================
# 20. JSON REPORT
# ================================================================

report = {
    "project": "SupplyChainAI",
    "notebook": "05_sentiment_analysis.ipynb",
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "data_source": DATA_SOURCE,
    "label_source": LABEL_SOURCE,
    "model": (
        "TF-IDF + LogisticRegression"
        if model is not None
        else None
    ),
    "metrics": metrics,
    "kpis": kpis,
    "limitations": [
        "Synthetic demo feedback is not real customer feedback.",
        "Rating-derived labels are weak supervision.",
        "The classifier is primarily designed for English text.",
        "Predicted probabilities are not calibrated confidence guarantees.",
        "Complaint topic extraction uses keyword rules.",
        "Model evaluation uses unique-text random splitting.",
        "Product-level alerts require sufficient review volume.",
        "No alerts are automatically sent to customers or employees.",
        "Human review is required before consequential supplier actions."
    ]
}

with open(
    REPORT_DIR / "sentiment_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        report,
        file,
        indent=2
    )

# ================================================================
# 21. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 75)
print("SUPPLYCHAINAI — SENTIMENT ANALYSIS COMPLETED")
print("=" * 75)

print(f"Feedback analyzed: {total_feedback:,}")
print(f"Positive: {positive_count:,}")
print(f"Neutral: {neutral_count:,}")
print(f"Negative: {negative_count:,}")
print(f"Negative rate: {negative_rate * 100:.2f}%")
print(f"Review alerts: {review_count:,}")

if model is not None:
    print(f"Model accuracy: {metrics['accuracy']:.3f}")
    print(f"Model macro F1: {metrics['macro_f1']:.3f}")

print("\nGenerated files:")
print("data/processed/sentiment_predictions.csv")
print("data/processed/negative_feedback_alerts.csv")
print("data/processed/product_sentiment_summary.csv")
print("reports/sentiment_analysis/sentiment_report.json")
print("reports/sentiment_analysis/sentiment_distribution.png")
print("reports/sentiment_analysis/negative_sentiment_trend.png")
print("reports/sentiment_analysis/complaint_topics.png")

if model is not None:
    print("models/sentiment_analysis/sentiment_classifier.joblib")

print("\nNext notebook:")
print("06_inventory_optimization.ipynb")

print("=" * 75)
